# 01 — Data profiling

Profiles the real downloaded Kaggle files against the assumptions in
[docs/DATA.md](../docs/DATA.md), per the roadmap gate: *"treat everything above as the
intended design, not a confirmed final schema."* Findings here already fed back into
`db/schema.sql` and the `RAW_COLUMN_MAP` constants in `etl/load_funds_postgres.py` /
`etl/load_esg_cosmos.py` — this notebook is the record of *why* those look the way they do.

Expects the four files described in [data/README.md](../data/README.md) to already be in
`data/raw/`.

In [1]:
from pathlib import Path

import pandas as pd

RAW = Path("..") / "data" / "raw"

mutual_funds = pd.read_csv(RAW / "morningstar_european_mutual_funds.csv", low_memory=False)
etfs = pd.read_csv(RAW / "morningstar_european_etfs.csv", low_memory=False)
holdings = pd.read_csv(RAW / "top100_etf_holdings.csv")
esg = pd.read_csv(RAW / "public_company_esg_ratings.csv")

mutual_funds.shape, etfs.shape, holdings.shape, esg.shape

((57603, 132), (9495, 132), (132182, 8), (722, 21))

## Tier 1 — Morningstar funds

Row counts came in close to `docs/DATA.md`'s estimate (~57.6k mutual funds / ~9.5k ETFs).
The **column set did not** match the original best-guess schema — see findings below.

In [2]:
print(f"{len(mutual_funds.columns)} columns in the real export (vs. ~16 assumed in the original schema draft)")
print("No management_company, domicile, sharpe_ratio, treynor_ratio, alpha, or beta column exists.")
print("Instead there's a much richer set: full sector/asset allocation, involvement flags,")
print("quarterly trailing returns back to 2015, and Morningstar's own portfolio E/S/G subscores.")

132 columns in the real export (vs. ~16 assumed in the original schema draft)
No management_company, domicile, sharpe_ratio, treynor_ratio, alpha, or beta column exists.
Instead there's a much richer set: full sector/asset allocation, involvement flags,
quarterly trailing returns back to 2015, and Morningstar's own portfolio E/S/G subscores.


In [3]:
cols = ["fund_size", "ongoing_cost", "sustainability_rank", "sustainability_score",
        "environmental_score", "social_score", "governance_score", "fund_trailing_return_ytd"]
(mutual_funds[cols].isnull().mean() * 100).round(1).rename("null_%")

fund_size                    1.8
ongoing_cost                 5.9
sustainability_rank         40.4
sustainability_score        40.0
environmental_score         45.0
social_score                45.0
governance_score            45.0
fund_trailing_return_ytd     3.9
Name: null_%, dtype: float64

`sustainability_rank`/`sustainability_score` (the claimed-rating side of
`docs/DATA.md#ground-truth-methodology`) are null on ~40% of mutual funds — expected, not
every fund carries a Morningstar Sustainability Rating. `sustainability_rank` is numeric
1-5 (globes), confirmed below, not a text label as originally assumed.

In [4]:
mutual_funds["sustainability_rank"].value_counts(dropna=False).sort_index()

sustainability_rank
1.0     2472
2.0     6321
3.0    12950
4.0     8734
5.0     3861
NaN    23265
Name: count, dtype: int64

### Domicile: derived from the ISIN prefix, not a column

No domicile field exists anywhere in the export. The ISIN's 2-letter country prefix is the
standard practical proxy — confirmed sane here: LU and IE dominate, matching their real-world
role as Europe's two main fund domiciles.

In [5]:
for label, df in [("Mutual funds", mutual_funds), ("ETFs", etfs)]:
    print(label)
    print((df["isin"].str[:2].value_counts(normalize=True) * 100).round(1).head(5))
    print()

Mutual funds
isin
LU    57.7
IE    21.3
GB    19.5
FR     0.4
IT     0.3
Name: proportion, dtype: float64

ETFs
isin
IE    51.3
LU    33.4
DE     6.5
FR     3.6
JE     1.9
Name: proportion, dtype: float64



### management_company: no column, ~44% recoverable from fund_name

Fund names loosely follow `"<Management Company> - <Fund Name> <Share Class>"`. Splitting on
`" - "` recovers a plausible company name on well under half the rows — kept as a best-effort,
nullable field (`etl/load_funds_postgres.py::_derive_management_company`), not an authoritative
one. GLEIF legal-name grounding remains the authoritative source for LU entity identity.

In [6]:
has_dash = mutual_funds["fund_name"].str.contains(" - ", regex=False)
print(f"{has_dash.mean()*100:.1f}% of mutual fund names contain a parseable \" - \" separator")
mutual_funds.loc[has_dash, "fund_name"].head(3).tolist()

47.2% of mutual fund names contain a parseable " - " separator


['Morgan Stanley Investment Funds - Global Bond Fund I',
 'Threadneedle (Lux) - American Select Class AU (USD Accumulation Shares)',
 'HSBC Global Investment Funds - Economic Scale Japan Equity PD']

## Tier 2 — ticker overlap between ETF holdings and company ESG ratings

This is the number that determines how much of the Tier 2 join
(`docs/DATA.md#two-tier-data-architecture`) is actually usable. It came in **much lower**
than assumed during scaffolding.

In [7]:
holding_tickers = set(holdings["holding_symbol"])
esg_tickers = set(esg["ticker"].str.upper())  # ESG file tickers are lowercase, holdings are uppercase

overlap = holding_tickers & esg_tickers
print(f"unique holding tickers across all 99 ETFs: {len(holding_tickers)}")
print(f"matched in the ~700-company ESG ratings file: {len(overlap)} ({100*len(overlap)/len(holding_tickers):.1f}%)")

unique holding tickers across all 99 ETFs: 4229
matched in the ~700-company ESG ratings file: 590 (14.0%)


In [8]:
coverage_by_etf = (
    holdings.assign(has_esg=holdings["holding_symbol"].isin(esg_tickers))
    .groupby("etf_symbol")["has_esg"]
    .mean()
    .mul(100)
    .round(1)
)
print(coverage_by_etf.describe())
print(f"\nETFs with 0% coverage: {(coverage_by_etf == 0).sum()} / {len(coverage_by_etf)}")
print(f"ETFs with >=50% coverage: {(coverage_by_etf >= 50).sum()} / {len(coverage_by_etf)}")
coverage_by_etf.sort_values(ascending=False).head(5)

count    99.000000
mean     30.679798
std      31.911258
min       0.000000
25%       0.000000
50%      16.000000
75%      62.350000
max      90.300000
Name: has_esg, dtype: float64

ETFs with 0% coverage: 35 / 99
ETFs with >=50% coverage: 32 / 99


etf_symbol
DIA     90.3
XLY     87.0
SPYV    80.5
XLV     80.3
IVE     79.5
Name: has_esg, dtype: float64

The 0%-coverage ETFs are bond funds (`AGG`, `BND`, `BIV`, `BSV`, ...) — a public-company
*equity* ESG ratings dataset has nothing to match against a bond holding. This isn't a data
quality problem, it's a category mismatch that narrows the Tier 2 *usable* subset further
than "Top 100 ETFs" implies: the greenwashing-risk model (Phase 2) should scope itself to
equity ETFs with reasonable coverage (e.g. the 32/99 at >=50%), not attempt all 99.

## Findings summary (real data)

- **Funds**: 57,603 mutual funds + 9,495 ETFs, no ticker collisions between the two files.
  132 columns each — far richer than assumed (full sector/asset allocation, involvement
  flags, quarterly returns back to 2015) but missing `management_company`, `domicile`, and
  the `sharpe_ratio`/`treynor_ratio`/`alpha`/`beta` fields the original schema draft assumed.
  `db/schema.sql` and `RAW_COLUMN_MAP` were updated to match.
- **Domicile**: not a column — derived from the ISIN country prefix. Sane distribution
  (LU/IE dominant for both fund types), matches real-world fund-domicile geography.
- **management_company**: not a column — best-effort parse from `fund_name`, ~44% coverage.
  Kept nullable; not treated as authoritative anywhere downstream.
- **Tier 2 overlap**: ~14% of unique ETF holding tickers match the ~700-company ESG ratings
  file overall (vs. an earlier synthetic-fixture estimate of ~79%, which was just illustrative
  scaffolding). Per-ETF coverage ranges 0-90.3%, median 16%. 35/99 ETFs have zero equity
  overlap (bond funds). The risk model should scope to the higher-coverage equity subset.
- **Holding names**: the holdings CSV stores names as a stringified dict scrape artifact
  (e.g. `"{'t': 'span', 'a': {}, 'c': ['Microsoft Corp']}"`) — parsed in `load_esg_cosmos.py`.
- **ESG score scale**: `total_score` in the company ratings file is NOT 0-100 — observed
  range ~600-1536 (sum of three subscores). Don't assume comparability with Tier 1's 0-100
  `sustainability_score` without normalizing first — a Phase 2 concern for the risk model.